<img src="../_static/pymt-logo-header-text.png">

## Coastline Evolution Model + Waves

* Link to this notebook: https://github.com/csdms/pymt/blob/master/notebooks/cem_and_waves.ipynb
* Install command: `$ conda install notebook pymt_cem`

This example explores how to use a BMI implementation to couple the Waves component with the Coastline Evolution Model component.

### Links
* [CEM source code](https://github.com/csdms/cem-old): Look at the files that have *deltas* in their name.
* [CEM description on CSDMS](http://csdms.colorado.edu/wiki/Model_help:CEM): Detailed information on the CEM model.

### Interacting with the Coastline Evolution Model BMI using Python

Some magic that allows us to view images within the notebook.

In [86]:
%matplotlib inline
import sys
import os

# Add the parent directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

%load_ext autoreload
%autoreload 2

import numpy as np



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Import the `Cem` class, and instantiate it. In Python, a model with a BMI will have no arguments for its constructor. Note that although the class has been instantiated, it's not yet ready to be run. We'll get to that later!

In [87]:
from pymt import models
from src.test_compose import compose
from src.test_compose import composeConfig

In [88]:
from decimal import Decimal, getcontext


In [89]:
cem, waves = models.Cem(), models.Waves()


/opt/conda/lib/python3.10/site-packages/model_metadata/model_parameter.py:160: UserWarning: 3650.0: floating point number passed as an integer parameter, value will be truncated to 3650
  return IntParameter(value, **kwds)


In [90]:

cem_compose = compose(waves,cem)

Even though we can't run our waves model yet, we can still get some information about it. *Just don't try to run it.* Some things we can do with our model are get the names of the input variables.

In [91]:
cem_compose.get_input_var_names()

('sea_surface_water_wave__height',
 'sea_surface_water_wave__period',
 'sea_shoreline_wave~incoming~deepwater__ashton_et_al_approach_angle_highness_parameter',
 'sea_shoreline_wave~incoming~deepwater__ashton_et_al_approach_angle_asymmetry_parameter',
 'sea_surface_water_wave__azimuth_angle_of_opposite_of_phase_velocity',
 'land_surface_water_sediment~bedload__mass_flow_rate',
 'land_surface__elevation',
 'model__time_step')

We can also get information about specific variables. Here we'll look at some info about wave direction. This is the main input of the Cem model. Notice that BMI components always use [CSDMS standard names](http://csdms.colorado.edu/wiki/CSDMS_Standard_Names). The CSDMS Standard Name for wave angle is,

    "sea_surface_water_wave__azimuth_angle_of_opposite_of_phase_velocity"

Quite a mouthful, I know. With that name we can get information about that variable and the grid that it is on (it's actually not a one).

OK. We're finally ready to run the model. Well not quite. First we initialize the model with the BMI **initialize** method. Normally we would pass it a string that represents the name of an input file. For this example we'll pass **None**, which tells Cem to use some defaults.

In [92]:




args = cem_compose.setup()
print(args)
cem_compose.initialize(args)

args = cem.setup(number_of_rows=100, number_of_cols=200, grid_spacing=200.0)
cem.initialize(*args)




('mergedConf.txt', '.')
Condition Initial 
Condition Initial 


Setting end time to 3650
CEM: trying to open file: bmi2Conf.txt
CEM: line: 50, 1000, 1000.0, 1

CEM: number of rows, columns: 50, 1000
*** Grid size is (0,0)
*** Requested size is (50,2000)
*** New grid size is (50,2000)
/opt/conda/lib/python3.10/site-packages/landlab/graph/graph.py:883: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  return self.ds.dims["patch"]
/opt/conda/lib/python3.10/site-packages/landlab/graph/graph.py:492: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  return self.ds.dims["link"]
CEM: trying to open file: cem.txt
CEM: line: 100, 200, 200.0, 1

CEM: number of rows, colu

Here I define a convenience function for plotting the water depth and making it look pretty. You don't need to worry too much about it's internals for this tutorial. It just saves us some typing later on.

In [94]:
def plot_coast(spacing, z):
    import matplotlib.pyplot as plt

    xmin, xmax = 0.0, z.shape[1] * spacing[1] * 1e-3
    ymin, ymax = 0.0, z.shape[0] * spacing[0] * 1e-3

    plt.imshow(z, extent=[xmin, xmax, ymin, ymax], origin="lower", cmap="ocean")
    plt.colorbar().ax.set_ylabel("Water Depth (m)")
    plt.xlabel("Along shore (km)")
    plt.ylabel("Cross shore (km)")


It generates plots that look like this. We begin with a flat delta (green) and a linear coastline (y = 3 km). The bathymetry drops off linearly to the top of the domain.

In [95]:
grid_id = cem.var_grid("sea_water__depth")
spacing = cem.grid_spacing(grid_id)
shape = cem.grid_shape(grid_id)
z = np.empty(shape)
cem_compose.get_value("sea_water__depth", z)
plot_coast(spacing, z)

Allocate memory for the sediment discharge array and set the discharge at the coastal cell to some value.

In [96]:
qs = np.zeros_like(z)
qs[0, 100] = 750

The CSDMS Standard Name for this variable is:

    "land_surface_water_sediment~bedload__mass_flow_rate"

You can get an idea of the units based on the quantity part of the name. "mass_flow_rate" indicates mass per time. You can double-check this with the BMI method function **get_var_units**.

In [97]:
cem.get_var_units("land_surface_water_sediment~bedload__mass_flow_rate")

/tmp/ipykernel_144783/2188898298.py:1: DeprecationWarning: Call to deprecated method get_var_units. (use var_units)
  cem.get_var_units("land_surface_water_sediment~bedload__mass_flow_rate")


'kg / s'

In [98]:
cem_compose.set_value(
    "sea_shoreline_wave~incoming~deepwater__ashton_et_al_approach_angle_asymmetry_parameter",
    0.3,
)
cem_compose.set_value(
    "sea_shoreline_wave~incoming~deepwater__ashton_et_al_approach_angle_highness_parameter",
    0.7,
)

cem_compose.set_value("sea_surface_water_wave__height", 2.0)
cem_compose.set_value("sea_surface_water_wave__period", 7.0)

cem_compose.get_value("sea_surface_water_wave__height")

print(waves.get_value("sea_surface_water_wave__azimuth_angle_of_opposite_of_phase_velocity"))



[-0.08994966]


Set the bedload flux and run the model.

In [99]:

qs = np.zeros_like(z)
qs[0, 100] = 750


for time in range(3000):
    cem_compose.set_value("land_surface_water_sediment~bedload__mass_flow_rate", qs)
    cem_compose.update()

    
cem_compose.get_value("sea_water__depth", z)


==== WaveAngle: -27.56:  MASS Percent: 0.0000:  Time Step: 0
==== WaveAngle: -57.77:  MASS Percent: 1.0029:  Time Step: 1000
==== WaveAngle: -57.67:  MASS Percent: 1.0056:  Time Step: 2000
==== WaveAngle: -86.41:  MASS Percent: 1.0085:  Time Step: 3000
==== WaveAngle: -46.26:  MASS Percent: 1.0112:  Time Step: 4000
==== WaveAngle: 55.74:  MASS Percent: 1.0142:  Time Step: 5000
==== WaveAngle: 78.37:  MASS Percent: 1.0172:  Time Step: 6000


xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is un

==== WaveAngle: -59.24:  MASS Percent: 1.0194:  Time Step: 7000
==== WaveAngle: -83.20:  MASS Percent: 1.0220:  Time Step: 8000


xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!

==== WaveAngle: 85.43:  MASS Percent: 1.0242:  Time Step: 9000
==== WaveAngle: -77.02:  MASS Percent: 1.0266:  Time Step: 10000
==== WaveAngle: -78.06:  MASS Percent: 1.0290:  Time Step: 11000
==== WaveAngle: -87.88:  MASS Percent: 1.0315:  Time Step: 12000
Loner fixbeach breakdown - mass disintegrated x: 42  y: 396
Loner fixbeach breakdown - mass disintegrated x: 42  y: 396
Loner fixbeach breakdown - mass disintegrated x: 42  y: 396
Loner fixbeach breakdown - mass disintegrated x: 42  y: 396
Loner fixbeach breakdown - mass disintegrated x: 42  y: 396
Loner fixbeach breakdown - mass disintegrated x: 42  y: 396
Loner fixbeach breakdown - mass disintegrated x: 42  y: 396
Loner fixbeach breakdown - mass disintegrated x: 42  y: 396
Loner fixbeach breakdown - mass disintegrated x: 42  y: 396
Loner fixbeach breakdown - mass disintegrated x: 42  y: 396
Loner fixbeach breakdown - mass disintegrated x: 42  y: 196
Loner fixbeach breakdown - mass disintegrated x: 42  y: 396
==== WaveAngle: -83.59

array([[-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       ...,
       [22.4, 22.4, 22.4, ..., 22.4, 22.4, 22.4],
       [22.6, 22.6, 22.6, ..., 22.6, 22.6, 22.6],
       [22.8, 22.8, 22.8, ..., 22.8, 22.8, 22.8]])

In [100]:
plot_coast(spacing, z)

Let's add another sediment source with a different flux and update the model.

In [101]:
qs[0, 150] = 500
for time in range(3750):
    cem_compose.set_value("land_surface_water_sediment~bedload__mass_flow_rate", qs)
    cem_compose.update()

cem_compose.get_value("sea_water__depth", z)

==== WaveAngle: -29.12:  MASS Percent: 1.0385:  Time Step: 15000
==== WaveAngle: 72.13:  MASS Percent: 1.0426:  Time Step: 16000
==== WaveAngle: 10.18:  MASS Percent: 1.0464:  Time Step: 17000
==== WaveAngle: 20.00:  MASS Percent: 1.0506:  Time Step: 18000
==== WaveAngle: -37.09:  MASS Percent: 1.0546:  Time Step: 19000
==== WaveAngle: -36.58:  MASS Percent: 1.0587:  Time Step: 20000
==== WaveAngle: -89.61:  MASS Percent: 1.0629:  Time Step: 21000
==== WaveAngle: -29.12:  MASS Percent: 1.0673:  Time Step: 22000
==== WaveAngle: 27.82:  MASS Percent: 1.0715:  Time Step: 23000
==== WaveAngle: 78.59:  MASS Percent: 1.0755:  Time Step: 24000
==== WaveAngle: -50.66:  MASS Percent: 1.0796:  Time Step: 25000


xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!

==== WaveAngle: -27.73:  MASS Percent: 1.0836:  Time Step: 26000
==== WaveAngle: -81.61:  MASS Percent: 1.0875:  Time Step: 27000


xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is uninitialized!xtest is uninitialized!ytest is un

==== WaveAngle: -47.79:  MASS Percent: 1.0912:  Time Step: 28000
==== WaveAngle: -58.65:  MASS Percent: 1.0943:  Time Step: 29000
==== WaveAngle: 69.97:  MASS Percent: 1.0983:  Time Step: 30000
Loner fixbeach breakdown - mass disintegrated x: 55  y: 394
Loner fixbeach breakdown - mass disintegrated x: 55  y: 194
Loner fixbeach breakdown - mass disintegrated x: 54  y: 395
==== WaveAngle: -54.90:  MASS Percent: 1.1019:  Time Step: 31000
==== WaveAngle: -73.88:  MASS Percent: 1.1055:  Time Step: 32000
==== WaveAngle: -28.63:  MASS Percent: 1.1095:  Time Step: 33000


array([[-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       ...,
       [22.4, 22.4, 22.4, ..., 22.4, 22.4, 22.4],
       [22.6, 22.6, 22.6, ..., 22.6, 22.6, 22.6],
       [22.8, 22.8, 22.8, ..., 22.8, 22.8, 22.8]])

In [102]:
plot_coast(spacing, z)

Here we shut off the sediment supply completely.

In [103]:
qs.fill(0.0)
for time in range(4000):
    cem_compose.set_value("land_surface_water_sediment~bedload__mass_flow_rate", qs)
    cem_compose.update()

cem_compose.get_value("sea_water__depth", z)

==== WaveAngle: -40.30:  MASS Percent: 1.1122:  Time Step: 34000
Loner fixbeach breakdown - mass disintegrated x: 79  y: 0
==== WaveAngle: -81.98:  MASS Percent: 1.1115:  Time Step: 35000
==== WaveAngle: -88.80:  MASS Percent: 1.1112:  Time Step: 36000
Loner fixbeach breakdown - mass disintegrated x: 77  y: 0
Loner fixbeach breakdown - mass disintegrated x: 77  y: 0
Loner fixbeach breakdown - mass disintegrated x: 77  y: 0
Loner fixbeach breakdown - mass disintegrated x: 77  y: 0
==== WaveAngle: -31.46:  MASS Percent: 1.1111:  Time Step: 37000
==== WaveAngle: -83.49:  MASS Percent: 1.1110:  Time Step: 38000
==== WaveAngle: -89.94:  MASS Percent: 1.1106:  Time Step: 39000
==== WaveAngle: 89.55:  MASS Percent: 1.1103:  Time Step: 40000
==== WaveAngle: -78.84:  MASS Percent: 1.1106:  Time Step: 41000
-- Empty Odd Over  xin: 71.94  yin: 395.50 xt:70 yt: 393 xint: 71.000000 yint: 393.917326 Meas: 150.00 Ang: -59.321699 Abs: 59.321699
-- Empty Odd Over  xin: 71.95  yin: 195.50 xt:70 yt: 193 

array([[-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       ...,
       [22.4, 22.4, 22.4, ..., 22.4, 22.4, 22.4],
       [22.6, 22.6, 22.6, ..., 22.6, 22.6, 22.6],
       [22.8, 22.8, 22.8, ..., 22.8, 22.8, 22.8]])

In [104]:
plot_coast(spacing, z)
